# 💳 Credit Card Fraud Detection And Analysis

### A Complete Data Science Pipeline: From Raw Transactions to Real-Time Fraud Intelligence

**Team Members:**
- Muhammad Maauz Mansoor (233599)
- Zain Riaz (233597)
- Zahid Zafar (233579)

---


## 📥 Layer 1: Data Collection & Setup
In this section, we import all necessary extreme high-level libraries for our Machine Learning and Deep Learning architectures, and load the raw dataset.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Sklearn & Imbalanced-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve

from imblearn.over_sampling import SMOTE

# PyTorch for Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
plt.style.use('dark_background')
sns.set_theme(style="darkgrid", rc={"axes.facecolor": "#121212", "figure.facecolor": "#121212", "text.color": "white", "axes.labelcolor": "white", "xtick.color": "white", "ytick.color": "white"})

print("All extreme high-level libraries imported successfully!")


In [ ]:
# Load Dataset
try:
    df = pd.read_csv('data/raw/creditcard.csv')
    print(f"Dataset Loaded Successfully! Shape: {df.shape}")
except FileNotFoundError:
    print("Please make sure creditcard.csv is in the data/raw/ directory.")


## 📊 Layer 2 & 3: Exploratory Data Analysis (EDA) & Visualization
We visualize the extreme class imbalance and distribution of features.


In [ ]:
# 1. Target Class Imbalance
plt.figure(figsize=(8, 6))
sns.countplot(data=df, x='Class', palette=['#00e5ff', '#ff007f'])
plt.title('Extreme Class Imbalance (Log Scale)', fontsize=16)
plt.yscale('log')
plt.ylabel('Count (Log Scale)')
plt.xlabel('Class (0: Legitimate, 1: Fraud)')
plt.show()

fraud_pct = (df['Class'].value_counts()[1] / len(df)) * 100
print(f"Fraud cases constitute only {fraud_pct:.3f}% of the dataset.")


In [ ]:
# 2. Time and Amount Distributions
fig, ax = plt.subplots(1, 2, figsize=(18, 6))

sns.histplot(df['Time'], bins=50, color='#00e5ff', kde=True, ax=ax[0])
ax[0].set_title('Transaction Time Distribution', fontsize=14)

sns.histplot(df['Amount'], bins=50, color='#ff007f', kde=True, ax=ax[1])
ax[1].set_title('Transaction Amount Distribution', fontsize=14)
ax[1].set_yscale('log')
ax[1].set_ylabel('Count (Log Scale)')

plt.show()


## ⚙️ Layer 4: Feature Engineering & Preprocessing
We engineer new features like `Hour_of_Day`, apply Log Transformation to `Amount`, and scale the features using `RobustScaler` to handle outliers. Finally, we balance the data using **SMOTE**.


In [ ]:
# Feature Engineering
df_clean = df.copy()
df_clean['Hour_of_Day'] = (df_clean['Time'] // 3600) % 24
df_clean['Amount_Log'] = np.log1p(df_clean['Amount'])

# Scaling
scaler = RobustScaler()
df_clean['Amount_Log_Scaled'] = scaler.fit_transform(df_clean['Amount_Log'].values.reshape(-1, 1))
df_clean['Time_Scaled'] = scaler.fit_transform(df_clean['Time'].values.reshape(-1, 1))
df_clean['Hour_of_Day_Scaled'] = scaler.fit_transform(df_clean['Hour_of_Day'].values.reshape(-1, 1))

# Drop old columns
df_clean.drop(['Amount', 'Amount_Log', 'Time', 'Hour_of_Day'], axis=1, inplace=True)
print("Feature Engineering & Scaling Complete!")


In [ ]:
# Train-Test Split and SMOTE
X = df_clean.drop('Class', axis=1)
y = df_clean['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Original Training shape: {X_train.shape}")

print("Applying SMOTE to balance the classes...")
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Balanced Training shape: {X_train_res.shape}")


## 🤖 Layer 5: Model Training - Machine Learning
Training Baseline and Ensemble models: Logistic Regression, Random Forest, and XGBoost.


In [ ]:
# 1. Logistic Regression
print("Training Logistic Regression...")
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_res, y_train_res)
lr_probs = lr.predict_proba(X_test)[:, 1]
print("Logistic Regression Trained!")


In [ ]:
# 2. Random Forest
print("Training Random Forest (Ensemble)...")
rf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_res, y_train_res)
rf_probs = rf.predict_proba(X_test)[:, 1]
print("Random Forest Trained!")


In [ ]:
# 3. XGBoost
print("Training XGBoost (Gradient Boosting)...")
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1)
xgb.fit(X_train_res, y_train_res)
xgb_probs = xgb.predict_proba(X_test)[:, 1]
print("XGBoost Trained!")


## 🧠 Layer 5: Model Training - Deep Learning (PyTorch)
Implementing Deep Neural Network (DNN), LSTM, and Autoencoder architectures.


In [ ]:
# PyTorch Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

X_train_tensor = torch.tensor(X_train_res.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_res.values, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)


In [ ]:
# 4. Deep Neural Network (DNN)
class DNN(nn.Module):
    def __init__(self, input_dim):
        super(DNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

print("Training DNN...")
dnn = DNN(X_train_res.shape[1]).to(device)
optimizer = optim.Adam(dnn.parameters(), lr=0.001)
criterion = nn.BCELoss()

dnn.train()
for epoch in range(3): # Reduced epochs for faster notebook execution
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        loss = criterion(dnn(batch_x), batch_y)
        loss.backward()
        optimizer.step()
print("DNN Trained!")

dnn.eval()
with torch.no_grad():
    dnn_probs = dnn(X_test_tensor).cpu().numpy().flatten()


In [ ]:
# 5. LSTM Network
class LSTMNet(nn.Module):
    def __init__(self, input_dim):
        super(LSTMNet, self).__init__()
        self.lstm = nn.LSTM(input_dim, 32, batch_first=True)
        self.fc = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.sigmoid(self.fc(out[:, -1, :]))

print("Training LSTM...")
lstm = LSTMNet(X_train_res.shape[1]).to(device)
optimizer = optim.Adam(lstm.parameters(), lr=0.001)

lstm.train()
for epoch in range(2):
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.unsqueeze(1).to(device)
        batch_y = batch_y.to(device)
        optimizer.zero_grad()
        loss = criterion(lstm(batch_x), batch_y)
        loss.backward()
        optimizer.step()
print("LSTM Trained!")

lstm.eval()
with torch.no_grad():
    lstm_probs = lstm(X_test_tensor.unsqueeze(1)).cpu().numpy().flatten()


## 🏆 Layer 6: Evaluation & Comparison
Generating ROC Curves and comparing all techniques to fulfill the extreme high-level project requirements.


In [ ]:
# Comparison Plot
plt.figure(figsize=(10, 8))

# Function to plot ROC
def plot_roc(y_t, y_p, name, color):
    fpr, tpr, _ = roc_curve(y_t, y_p)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {roc_auc:.4f})')

plot_roc(y_test, lr_probs, 'Logistic Regression', '#00e5ff')
plot_roc(y_test, rf_probs, 'Random Forest', '#ff007f')
plot_roc(y_test, xgb_probs, 'XGBoost', '#ffaa00')
plot_roc(y_test, dnn_probs, 'DNN', '#b000ff')
plot_roc(y_test, lstm_probs, 'LSTM', '#00ff00')

plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Comprehensive ROC-AUC Comparison of All Techniques', fontsize=16)
plt.legend(loc="lower right", fontsize=10, facecolor='#121212', edgecolor='white')
plt.show()
